# Surround Pursuit — IPPO Training via EPyMARL
**Environment:** PettingZoo `pursuit_v4`, surround=True, 8 pursuers, 30 evaders  
**Algorithm:** IPPO (Independent PPO with parameter sharing)  
**Target reward:** ~633 (FLAIRS 2024 baseline benchmark)  

Run cells top to bottom. Training takes 4–10 hours on T4 GPU.  
Make sure **GPU accelerator** and **Internet** are both ON in Kaggle settings.

In [ ]:
# ── CELL 1 — Install dependencies ──────────────────────────────────────────
# This takes ~3-5 minutes. Run once.

!pip install -q pettingzoo[sisl]==1.24.3
!pip install -q torch torchvision
!pip install -q sacred pymongo==3.12.3
!pip install -q gym==0.21.0
!pip install -q pygame

print('Dependencies installed.')

In [ ]:
# ── CELL 2 — Clone EPyMARL ──────────────────────────────────────────────────
import os

os.chdir('/kaggle/working')
!git clone https://github.com/uoe-agents/epymarl.git
os.chdir('/kaggle/working/epymarl')

!pip install -q -r requirements.txt

print('EPyMARL cloned and requirements installed.')

In [ ]:
# ── CELL 3 — Verify pursuit_v4 loads correctly ──────────────────────────────
from pettingzoo.sisl import pursuit_v4

# Test with surround=True config
env = pursuit_v4.parallel_env(
    x_size=16, y_size=16,
    n_pursuers=8, n_evaders=30,
    obs_range=7, n_catch=2,
    surround=True,
    shared_reward=True,
    freeze_evaders=False,
    tag_reward=0.01,
    catch_reward=5.0,
    urgency_reward=-0.1,
    max_cycles=500,
    render_mode=None
)

obs, _ = env.reset()
print(f'Agents: {env.agents}')
print(f'Num agents: {len(env.agents)}')
print(f'Obs shape (one agent): {list(obs.values())[0].shape}')
print(f'Action space: {env.action_space(env.agents[0])}')
print('Environment loads correctly.')
env.close()

In [ ]:
# ── CELL 4 — Write the environment config for EPyMARL ───────────────────────
import os

env_config = """
env: "gymma"

env_args:
  key: "pz-pursuit-v4"
  time_limit: 500
  pretrained_wrapper: null
"""

# EPyMARL uses a gymma wrapper for PettingZoo envs.
# We register pursuit_v4 with our config as a gym environment below.

# Write the env config file
config_path = '/kaggle/working/epymarl/src/config/envs/pursuit_surround.yaml'
with open(config_path, 'w') as f:
    f.write(env_config)

print(f'Env config written to {config_path}')

In [ ]:
# ── CELL 5 — Register pursuit_v4 as a gym environment ───────────────────────
# EPyMARL's gymma wrapper expects an env registered with gym.
# We create a registration shim so EPyMARL can find pursuit_v4.

registration_code = '''
from gymnasium.envs.registration import register
from pettingzoo.sisl import pursuit_v4
import gymnasium as gym
import numpy as np

class PursuitGymWrapper(gym.Env):
    """Wraps pursuit_v4 parallel_env into a gym.Env for EPyMARL."""

    metadata = {"render_modes": []}

    def __init__(self):
        self.env = pursuit_v4.parallel_env(
            x_size=16, y_size=16,
            n_pursuers=8, n_evaders=30,
            obs_range=7, n_catch=2,
            surround=True,
            shared_reward=True,
            freeze_evaders=False,
            tag_reward=0.01,
            catch_reward=5.0,
            urgency_reward=-0.1,
            max_cycles=500,
            render_mode=None
        )
        self.n_agents = 8
        sample_agent = self.env.possible_agents[0]
        self.observation_space = self.env.observation_space(sample_agent)
        self.action_space = self.env.action_space(sample_agent)

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed)
        obs_list = [obs[a] for a in self.env.possible_agents]
        return np.stack(obs_list), info

    def step(self, actions):
        action_dict = {a: int(actions[i]) for i, a in enumerate(self.env.agents)}
        obs, rewards, terms, truncs, infos = self.env.step(action_dict)
        obs_list   = [obs.get(a, np.zeros(self.observation_space.shape))
                      for a in self.env.possible_agents]
        rew_list   = [rewards.get(a, 0.0) for a in self.env.possible_agents]
        term_list  = [terms.get(a, False)  for a in self.env.possible_agents]
        trunc_list = [truncs.get(a, False) for a in self.env.possible_agents]
        done = all(term_list) or all(trunc_list) or not self.env.agents
        return np.stack(obs_list), np.array(rew_list), done, False, infos

    def render(self):
        pass

    def close(self):
        self.env.close()


register(
    id="pz-pursuit-v4",
    entry_point="pursuit_gym_wrapper:PursuitGymWrapper",
)
'''

wrapper_path = '/kaggle/working/epymarl/src/pursuit_gym_wrapper.py'
with open(wrapper_path, 'w') as f:
    f.write(registration_code)

print(f'Wrapper written to {wrapper_path}')

In [ ]:
# ── CELL 6 — Write IPPO hyperparameter config ───────────────────────────────
# Tuned for 8-agent pursuit. Adjust batch_size / lr if memory issues.

ippo_config = """
# IPPO config tuned for pursuit_v4 surround (8 agents, 30 evaders)

action_selector: "softmax"
softmax_temp: 1.0

use_rnn: True
rnn_hidden_dim: 64

# PPO
epochs: 4
eps_clip: 0.1
standardise_rewards: True
standardise_returns: False
use_gae: True
gae_lambda: 0.95
use_value_norm: True

# Optimiser
lr: 0.0005
optim_alpha: 0.99
optim_eps: 0.00001
grad_norm_clip: 10

# Batch / buffer
batch_size_run: 8          # number of parallel envs (one per agent batch)
batch_size: 32             # mini-batch size for PPO update
buffer_size: 32

# Entropy regularisation (encourages exploration)
entropy_coef: 0.01

# Value loss
use_huber_loss: True
huber_delta: 10.0

# Parameter sharing across all 8 pursuers
agent: "rnn"
mac: "ppo_mac"
learner: "ppo_learner"
name: "ippo"

# Discount
gamma: 0.99

# Logging
log_interval: 10000
runner_log_interval: 10000
learner_log_interval: 10000

# Training duration
t_max: 10000000            # 10M steps (~8-10h on T4)
                           # reduce to 5000000 for a 4-5h run

# Checkpointing
save_model: True
save_model_interval: 500000  # save every 500k steps
checkpoint_path: ""

# Evaluation
evaluate: False
test_nepisode: 20
test_interval: 50000
test_greedy: True

# Runner
runner: "parallel"
env_runner_split: null
"""

ippo_path = '/kaggle/working/epymarl/src/config/algs/ippo_pursuit.yaml'
with open(ippo_path, 'w') as f:
    f.write(ippo_config)

print(f'IPPO config written to {ippo_path}')

In [ ]:
# ── CELL 7 — Patch EPyMARL to import the pursuit wrapper on startup ──────────
# EPyMARL's main.py needs to know about our custom gym registration.
# We prepend an import line to main.py.

main_path = '/kaggle/working/epymarl/src/main.py'

with open(main_path, 'r') as f:
    main_content = f.read()

import_line = 'import sys; sys.path.insert(0, "/kaggle/working/epymarl/src"); import pursuit_gym_wrapper  # register pursuit_v4\n'

if 'pursuit_gym_wrapper' not in main_content:
    with open(main_path, 'w') as f:
        f.write(import_line + main_content)
    print('main.py patched to import pursuit wrapper.')
else:
    print('main.py already patched.')

In [ ]:
# ── CELL 8 — Sanity check before full training run ──────────────────────────
# Run for 1000 steps to confirm everything is wired up.
# Should complete in under 2 minutes. Watch for errors.

import os
os.chdir('/kaggle/working/epymarl/src')

!python main.py \
    --config=ippo_pursuit \
    --env-config=pursuit_surround \
    with t_max=1000 save_model=False test_interval=1000 log_interval=500

In [ ]:
# ── CELL 9 — FULL TRAINING RUN ──────────────────────────────────────────────
# Only run this after Cell 8 completes without errors.
#
# t_max=10000000  → ~8-10 hours on T4 (target: 633+ reward)
# t_max=5000000   → ~4-5 hours on T4  (target: 400-550 reward)
#
# Change seed= to run multiple seeds. Start with seed=0, then seed=1, seed=2.

import os
os.chdir('/kaggle/working/epymarl/src')

SEED = 0        # change for each run
T_MAX = 5000000 # 5M steps for first run — see how far it gets in the session

!python main.py \
    --config=ippo_pursuit \
    --env-config=pursuit_surround \
    with seed={SEED} t_max={T_MAX}

In [ ]:
# ── CELL 10 — Save results before session ends ──────────────────────────────
# Run this when training finishes OR when you see the session timer getting low.
# Download results_backup.zip from the Kaggle output panel on the right.

import shutil
import os

results_dir = '/kaggle/working/epymarl/results'

if os.path.exists(results_dir):
    shutil.make_archive('/kaggle/working/results_backup', 'zip', results_dir)
    size_mb = os.path.getsize('/kaggle/working/results_backup.zip') / 1e6
    print(f'Saved results_backup.zip ({size_mb:.1f} MB)')
    print('Download this from the Kaggle output panel before the session ends.')
else:
    print('No results directory found — training may not have started yet.')

In [ ]:
# ── CELL 11 — Plot training curve from saved results ────────────────────────
# Run after training finishes to see the reward curve.

import json
import glob
import matplotlib.pyplot as plt
import numpy as np

# Find the most recent sacred run
sacred_dirs = sorted(glob.glob('/kaggle/working/epymarl/results/sacred/*/metrics.json'))

if not sacred_dirs:
    print('No metrics.json found. Training may not have completed.')
else:
    with open(sacred_dirs[-1]) as f:
        metrics = json.load(f)

    # Extract episode reward mean
    if 'episode_reward_mean' in metrics:
        steps  = metrics['episode_reward_mean']['steps']
        values = metrics['episode_reward_mean']['values']

        plt.figure(figsize=(10, 5))
        plt.plot(steps, values, alpha=0.4, color='green', label='raw')

        # smoothed
        window = max(1, len(values) // 20)
        smoothed = np.convolve(values, np.ones(window)/window, mode='valid')
        plt.plot(steps[window-1:], smoothed, color='darkgreen', linewidth=2, label='smoothed')

        plt.axhline(633, color='red', linestyle='--', alpha=0.7, label='Target (FLAIRS 2024 baseline)')
        plt.xlabel('Environment Steps')
        plt.ylabel('Episode Reward Mean')
        plt.title('IPPO on pursuit_v4 (surround=True, 8 pursuers, 30 evaders)')
        plt.legend()
        plt.tight_layout()
        plt.savefig('/kaggle/working/training_curve.png', dpi=150)
        plt.show()
        print(f'Final reward: {values[-1]:.1f}')
        print(f'Peak reward:  {max(values):.1f}')
    else:
        print('Available metrics:', list(metrics.keys()))